In this assignment, you will be implementing a GPT model and train it using CLM objective.
 * If you get stuck at something or need more clarrifications, you may refer to : https://github.com/karpathy/minGPT/blob/master/mingpt/model.py

 * We will be using ReLU activation function instead of GELU.

 * As usual, let us install the required libraries

 * **Note** that if you are not getting the exact loss values as mentioned in this notebook, that is absolutely fine. Just see whether your implementation overfits the given toy-and-tiny paragraph!

# Installation


In [186]:
!pip install torchdata==0.6.0 # to be compatible with torch 2.0
!pip install portalocker==2.0.0

* See [here](https://github.com/pytorch/text) for compatability

In [187]:
!pip install -U torchtext==0.15.1

# Imports

In [188]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

#text lib
import torchtext

# tokenizer
from torchtext.data.utils import get_tokenizer

#build vocabulary
from torchtext.vocab import vocab
from torchtext.vocab import build_vocab_from_iterator

# get input_ids (numericalization)
from torchtext.transforms import VocabTransform

# get embeddings
from torch.nn import Embedding

from  pprint import pprint
from yaml import safe_load
import copy
import numpy as np

In [189]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Load the dataset for LM modeling

 * We use a simple tokenizer and put

In [190]:
batch_size = 10

In [191]:
class Tokenizer(object):

  def __init__(self,text):
    self.text = text
    self.word_tokenizer = get_tokenizer(tokenizer="basic_english",language='en')
    self.vocab_size = None

  def get_tokens(self):
    for sentence in self.text.strip().split('\n'):
      yield self.word_tokenizer(sentence)

  def build_vocab(self):
    v = build_vocab_from_iterator(self.get_tokens(),
                                  min_freq=1,specials=['<unk>','<start>','<end>'])
    v.set_default_index(v['<unk>']) # index of OOV
    self.vocab_size = len(v)
    return v

  def token_ids(self):
    v = self.build_vocab()
    vt = VocabTransform(v)
    num_tokens = len(self.word_tokenizer(self.text))
    max_seq_len = np.ceil(num_tokens/batch_size)
    data = torch.zeros(size=(1,num_tokens))
    data = vt(self.word_tokenizer(self.text))
    data = torch.tensor(data,dtype=torch.int64)
    return data.reshape(batch_size,torch.tensor(max_seq_len,dtype=torch.int64))



In [192]:
text = """Best known for the invention of Error Correcting Codes, he was a true polymath who applied his mathematical and problem-solving skills to numerous disciplines.
Reflecting on the significant benefits I received from Hamming, I decided to develop a tribute to his legacy. There has not been a previous biography of Hamming, and the few articles about him restate known facts and assumptions and leave us with open questions.
One thought drove me as I developed this legacy project: An individual's legacy is more than a list of their attempts and accomplishments. Their tribute should also reveal the succeeding generations they inspired and enabled and what each attempted and achieved.
This book is a unique genre containing my version of a biography that intertwines the story "of a life" and a multi-player memoir with particular events and turning points recalled by those, including me, who he inspired and enabled.
Five years of research uncovered the people, places, opportunities, events, and influences that shaped Hamming. I discovered unpublished information, stories, photographs, videos, and personal remembrances to chronicle his life, which helped me put Hamming's
legacy in the context I wanted.The result demonstrates many exceptional qualities, including his noble pursuit of excellence and helping others. Hamming paid attention to the details, his writings continue to influence, and his guidance is a timeless gift to the world.
This biography is part of """

In [193]:
Tk = Tokenizer(text)

In [194]:
x_raw = Tk.token_ids()
print(x_raw.shape)

torch.Size([10, 26])


In [195]:
# let us display the first 10 tokens of the vocabulary
v = Tk.build_vocab()
pprint(v.vocab.get_itos()[0:10])

['<unk>', '<start>', '<end>', ',', 'and', '.', 'the', 'a', 'of', 'to']


* Create the input_ids and Labels from the raw input sequence

In [196]:
bs,raw_seq_len = x_raw.shape
x = torch.empty(size=(bs,raw_seq_len+2),dtype=torch.int64)
x[:,1:-1] =x_raw

# insert the index of special tokens
x[:,0] = torch.full(size=(1,batch_size),fill_value=v.vocab.get_stoi()['<start>'])
x[:,-1] = torch.full(size=(1,batch_size),fill_value=v.vocab.get_stoi()['<end>'])

#Quickly check implem
v = Tk.build_vocab()
words = []
for idx in x[0,:]:
  words.append(v.vocab.get_itos()[idx.item()])
print(' '.join(words))

<start> best known for the invention of error correcting codes , he was a true polymath who applied his mathematical and problem-solving skills to numerous disciplines . <end>


In [197]:
# labels are just the input_ids shifted by right
bs,seq_len = x.shape
y = torch.empty(size=(bs,seq_len),dtype=torch.int64)
y[:,0:-1] = copy.deepcopy(x[:,1:])

#ignore the index of padded tokens while computing loss
y[:,-1] = torch.full(size=(1,batch_size),fill_value=-100)

# Configuration

In [198]:
vocab_size = Tk.vocab_size
seq_len = x.shape[1]
embed_dim = 32
dmodel = embed_dim
dq = torch.tensor(4)
dk = torch.tensor(4)
dv = torch.tensor(4)
heads = torch.tensor(8)
d_ff = 4*dmodel

* Define all the sub-layers (mhma,ffn) in the transformer blocks
* Seed for $W_Q,W_K,W_V,W_O$, 43, 44 and 45, 46, respectively
* Seed for ffn $W_1,W_2$,  47 and 48. There are no biases
* Seed for output layer 49

In [199]:
import math
class MHMA(nn.Module):
  def __init__(self, heads, dmodel, dq, dk, dv):
    super(MHMA, self).__init__()
    self.d_model = dmodel
    self.heads = heads
    self.W_q = nn.Parameter(torch.randn((heads, dmodel, dq),generator = torch.manual_seed(43)))
    self.W_k = nn.Parameter(torch.randn((heads, dmodel, dk),generator = torch.manual_seed(44)))
    self.W_v = nn.Parameter(torch.randn((heads, dmodel, dv),generator = torch.manual_seed(45)))
    self.W_o = nn.Parameter(torch.randn((dmodel, dmodel),generator = torch.manual_seed(46)))
    self.mask = (torch.triu(torch.ones(seq_len,seq_len)) == 1).transpose(0,1)
    self.mask = self.mask.float().masked_fill(self.mask == 0, float('-inf')).masked_fill(self.mask == 1, float(0.0))

  def forward(self, Q, K, V):
    BS, T, _ = Q.shape
    #print(Q.shape, self.W_q.shape)
    Q = torch.einsum('BTM, HMQ -> BHTQ', Q, self.W_q)
    K = torch.einsum('BTM, HMK -> BHTK', K, self.W_k)
    V = torch.einsum('BTM, HMV -> BHTV', V, self.W_v)
    attn_score = torch.matmul(F.softmax((torch.matmul(Q,torch.transpose(K, -2, -1)) + self.mask)/math.sqrt(dq), dim = -1), V)
    combined_attn = attn_score.permute(0,2,1,3).contiguous().view(BS, T, -1)
    out = torch.matmul(combined_attn, self.W_o)
    #print(f'Output after MHMA : {out.shape}')
    return out


class FFN(nn.Module):
  def __init__(self, dmodel, d_ff):
    super(FFN, self).__init__()
    self.W1 = nn.Parameter(torch.randn((dmodel, d_ff), generator = torch.manual_seed(47))) #Weights
    #self.b1 = nn.Parameter(torch.randn((d_ff), generator = torch.manual_seed(10))) #Bias
    self.W2 = nn.Parameter(torch.randn((d_ff, dmodel), generator = torch.manual_seed(48))) #Weights
    #self.b2 = nn.Parameter(torch.randn((dmodel), generator = torch.manual_seed(10))) #Bias
    self.relu = nn.ReLU()

  def forward(self, x):
    out = torch.einsum('BTM, MH -> BTH', x, self.W1)# + self.b1
    out = self.relu(out)
    out = torch.einsum('BTM, MH -> BTH', out, self.W2)# + self.b2
    #print(f'Output after FFN : {out.shape}')
    return out

class PredictionHead(nn.Module):
  def __init__(self, dmodel, trgt_vocab_size):
    super(PredictionHead, self).__init__()
    self.W = nn.Parameter(torch.randn((dmodel, trgt_vocab_size), generator = torch.manual_seed(49)))
    #self.b = nn.Parameter(torch.randn((trgt_vocab_size), generator = torch.manual_seed(10)))

  def forward(self, x):
    out = torch.matmul(x, self.W)# + self.b
    #print(f'Output shape after Output Layer: {out.shape}')
    return out


class PositionalEncoding(nn.Module):
  def __init__(self,d_model, max_seq_len = 512):
      super(PositionalEncoding, self).__init__()

      #compute it in the log space
      pe = torch.zeros(max_seq_len, dmodel)
      position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
      div_term = torch.exp(torch.arange(0, dmodel, 2).float() * (-torch.log(torch.tensor(10000.0)) / dmodel))
      pe[:, 0::2] = torch.sin(position * div_term)
      pe[:, 1::2] = torch.cos(position * div_term)
      self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
      # add positional embedding
      x = x + self.pe[:, :x.size(1)]
      return x


In [200]:
class DecoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads,mask=None):
    super(DecoderLayer,self).__init__()
    self.mhma = MHMA(heads, dmodel,dq,dk,dv)#,mask=None)
    self.layer_norm_1 = torch.nn.LayerNorm(dmodel)
    self.layer_norm_2 = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,y):
    """
    your code goes here
    """
    out = self.mhma(y, y, y)
    out1 = self.layer_norm_1(out + y)
    out1 = self.ffn(out)
    out = self.layer_norm_2(out + out1)
    return out

In [201]:
class Embed(nn.Module):

  def __init__(self,vocab_size,embed_dim):
    super(Embed,self).__init__()
    self.embed = nn.Embedding(num_embeddings = vocab_size, embedding_dim = dmodel, _weight = torch.randn((vocab_size, dmodel), generator=torch.manual_seed(70)) ) #None # seed 70
    self.pe = PositionalEncoding(dmodel)

  def forward(self,x):
    out = self.pe(self.embed(x))
    return out

In [202]:
class Decoder(nn.Module):

  def __init__(self,vocab_size,dmodel,dq,dk,dv,d_ff,heads,mask, num_layers=1):
    super(Decoder,self).__init__()
    self.embed_lookup = Embed(vocab_size,embed_dim)
    self.dec_layers = nn.ModuleList(copy.deepcopy(DecoderLayer(dmodel,dq,dk,dv,d_ff,heads,mask)) for i in range(num_layers))
    self.predict = PredictionHead(dmodel,vocab_size)

  def forward(self,input_ids):
    out = self.embed_lookup(input_ids)
    for dec_layer in self.dec_layers:
      out = dec_layer(out)
    out = self.predict(out)

    return out

In [203]:
model = Decoder(vocab_size,dmodel,dq,dk,dv,d_ff,heads,mask=None)

In [204]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [205]:
def train(input_ids,labels,epochs=1000):
  loss_trace = []
  for epoch in range(epochs):
    out = model(input_ids)
    out = out.view(-1,vocab_size)
    target = labels.view(-1)
    loss = criterion(out, target.type(torch.LongTensor)) # edit this
    if (epoch+1)%1000 == 0:
      loss_trace.append(loss.item())
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
  print(loss_trace)


In [206]:
# run the model for 10K epochs
train(x,y,10000)

[4.269568920135498, 3.7445459365844727, 3.0982961654663086, 2.365241765975952, 1.6131936311721802, 0.9906038641929626, 0.5601183772087097, 0.3326006829738617, 0.21868473291397095, 0.16382552683353424]


The loss is about 0.09 after 10K epochs

# Generate text

In [227]:
@torch.inference_mode()
def generate(model,prompt='<start>',max_words=10):
  with torch.no_grad():
    tar_token_ids = torch.zeros((1,max(max_words,seq_len)), dtype = torch.int)
    if type(prompt) == str:
      prompt = [prompt]
    for i in range(len(prompt)):
      tar_token_ids[:, i] = v.get_stoi()[prompt[i]]
    for item in range(len(prompt)-1, max_words-1):
      if item < seq_len:
        input_tokens = tar_token_ids[:,:seq_len]
        #print(f'Within - {input_tokens.shape}')
      else:
        input_tokens = tar_token_ids[:,item-seq_len:item] #To handle sequence generation beyond context window using sliding window
        #print(f'Beyond - {input_tokens.shape}')
      out = model(input_tokens)
      input = out.view(-1,vocab_size)
      probs = F.softmax(input, dim = -1)
      pred = torch.argmax(probs, dim = -1)
      if item < seq_len:
        tar_token_ids[:, item + 1] = pred[item]
      else:
        tar_token_ids[:, item + 1] = pred[-1] #To handle sequence generation beyond context window
  tar_token_ids = tar_token_ids.squeeze()
  for i in range(tar_token_ids.shape[0]):
    if (i < max_words):
      print(v.get_itos()[tar_token_ids[i]], end=' ')
  return

In [229]:
generate(model,prompt='<start>',max_words=25)

<start> biography of hamming , and the few articles about him restate known facts and assumptions and leave us with open questions . one thought 

* Note the model has memorized the sentence from the training set. Given the start token, if your implementation reproduce a sentence as is in the training set, then your implementation is likely to be correct.
* Suppose the prompt is `<start> best known`, then we expect the model to produce the first sentence as is

In [ ]:
generate(model,prompt=['<start>','best','known'],max_words=25)

<start> best known for the invention of error correcting codes , he was a true polymath who applied his mathematical and problem-solving skills to numerous 

* Change the prompt

In [230]:
generate(model,prompt=['<start>','reflecting','on'],max_words=25)

<start> reflecting on the significant benefits i received from hamming , i decided to develop a tribute to his legacy . there has not been 